# Conditional WGAN-GP — Hyperparameter Search with Optuna
This notebook extends the `conditional_wgan_gp` notebook with an Optuna study that searches over
`lr_G`, `lr_D`, `lambda_fm_1`, `lambda_fm_2`, and `lambda_entropy`.
After the search we retrain a final model with the best hyperparameters and inspect the results.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import optuna
from optuna.visualization.matplotlib import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate,
)
from synthetic_generation.gan.models import Generator, Discriminator, OutputHead
from synthetic_generation.gan.training import train_wgan_gp

optuna.logging.set_verbosity(optuna.logging.WARNING)

#### Creating Data
We use the same ring-of-8-clusters dataset as the base notebook. The conditional vector
concatenates scaled cluster centers with one-hot encoded labels.

In [ ]:
def make_gaussian_mixture(
    n_samples=50_000,
    centers=8,
    radius=5.0,
    std=0.5,
    seed=0
):
    """
    Parameters
    ----------
    n_samples : int
        number of samples
    centers : int
        number of disk clusters
    radius : float
        radius of ring containing centers of the clusters
    std : float
        standard deviation for distance from center of disk for a point
    seed : int
        random seed to use
    Returns
    -------
    tuple
        array of sample points, array of cluster centers for each point,
        array of labels for each cluster
    """
    rng = np.random.default_rng(seed)
    angles = np.linspace(0, 2 * np.pi, centers, endpoint=False)
    means = np.stack([
        radius * np.cos(angles),
        radius * np.sin(angles),
    ], axis=1)

    samples, sample_labels, sample_centers = [], [], []
    for _ in range(n_samples):
        k = rng.integers(0, centers)
        sample_labels.append(k)
        sample_centers.append(means[k])
        samples.append(rng.normal(loc=means[k], scale=std, size=2))

    return (
        np.array(samples, dtype=np.float32),
        np.array(sample_labels),
        np.array(sample_centers, dtype=np.float32),
    )

In [ ]:
X, c_label, c_center = make_gaussian_mixture()

# one-hot encode labels
one_hot = torch.nn.functional.one_hot(torch.tensor(c_label), num_classes=8).float()

# scale X and centers to [-1, 1]
X_min, X_max = X.min(axis=0), X.max(axis=0)
X_scaled = 2 * (X - X_min) / (X_max - X_min) - 1
c_center_scaled = 2 * (c_center - X_min) / (X_max - X_min) - 1

# combined conditional: scaled centers + one-hot labels
c = torch.cat([torch.tensor(c_center_scaled), one_hot], dim=1)
X_tensor = torch.tensor(X_scaled)

CONDITIONAL_DIM = c.shape[1]   # 2 + 8 = 10
print(f"X_tensor: {X_tensor.shape}, c: {c.shape}")

#### Evaluation Metric
We need a differentiable-free quality score to drive the Optuna objective.
After training we generate one sample per real data-point (using the same conditionals),
then measure the **mean squared distance from each generated point to its assigned cluster center**
(the center is read directly from the conditional vector, so no extra inference is needed).
Lower = the generator is placing points tightly around the correct centers.

In [ ]:
@torch.no_grad()
def cluster_mse(G: Generator, c_eval: torch.Tensor, c_center_eval: np.ndarray, n: int = 4000) -> float:
    """
    Generate *n* samples (using the first *n* rows of c_eval) and return
    the mean squared distance of each sample to its true cluster center
    (unscaled back to the original coordinate system).

    Parameters
    ----------
    G : Generator
    c_eval : torch.Tensor  (N, conditional_dim)
    c_center_eval : np.ndarray  (N, 2)  — unscaled centers
    n : int  — how many samples to evaluate on
    """
    G.eval()
    device = next(G.parameters()).device
    c_in = c_eval[:n].to(device)
    x_fake = G.generate_samples(num_samples=n, c=c_in).cpu().numpy()
    # undo [-1,1] scaling
    x_fake_unscaled = (x_fake + 1) / 2 * (X_max - X_min) + X_min
    centers_n = c_center_eval[:n]
    mse = float(np.mean(np.sum((x_fake_unscaled - centers_n) ** 2, axis=1)))
    G.train()
    return mse

#### Optuna Objective
Each trial:
1. Samples hyperparameters from the search space
2. Builds fresh G and D
3. Trains for `SEARCH_EPOCHS` epochs (shorter than the full run)
4. Returns `cluster_mse` — Optuna minimises this

In [ ]:
SEARCH_EPOCHS = 60   # quick enough for many trials; bump up if you have a GPU
N_TRIALS      = 30


def objective(trial: optuna.Trial) -> float:
    # --- hyperparameter search space ---
    lr_G          = trial.suggest_float("lr_G",          1e-5, 1e-3, log=True)
    lr_D          = trial.suggest_float("lr_D",          1e-5, 1e-3, log=True)
    lambda_fm_1   = trial.suggest_float("lambda_fm_1",   0.0,  1.0)
    lambda_fm_2   = trial.suggest_float("lambda_fm_2",   0.0,  1.0)
    lambda_entropy = trial.suggest_float("lambda_entropy", 0.0, 0.5)

    torch.manual_seed(trial.number)  # reproducible per trial

    heads = [OutputHead(dim=2, activation=nn.Tanh, decode=nn.Tanh, name="scaled_output")]
    G = Generator(
        noise_dim=2,
        num_hidden_layers=2,
        output_heads=heads,
        hidden_dims=(128, 128),
        conditional_dim=CONDITIONAL_DIM,
    )
    D = Discriminator(
        feature_dim=2,
        num_hidden_layers=2,
        hidden_dims=(128, 128),
        conditional_dim=CONDITIONAL_DIM,
    )

    train_wgan_gp(
        G=G, D=D,
        X=X_tensor, c=c,
        lr_G=lr_G, lr_D=lr_D,
        lambda_fm_1=lambda_fm_1,
        lambda_fm_2=lambda_fm_2,
        lambda_entropy=lambda_entropy,
        epochs=SEARCH_EPOCHS,
    )

    return cluster_mse(G, c, c_center)

#### Run the Study

In [ ]:
sampler = optuna.samplers.TPESampler(seed=0)
study = optuna.create_study(direction="minimize", sampler=sampler)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

#### Best Parameters

In [ ]:
best = study.best_trial
print(f"Best trial: #{best.number}  cluster_mse = {best.value:.6f}")
print("Best hyperparameters:")
for k, v in best.params.items():
    print(f"  {k}: {v:.6g}")

#### Optuna Visualizations

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 5))
plot_optimization_history(study, ax=axs[0])
axs[0].set_title("Optimization History")
plot_param_importances(study, ax=axs[1])
axs[1].set_title("Hyperparameter Importances")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
plot_parallel_coordinate(study, ax=ax)
ax.set_title("Parallel Coordinate Plot")
plt.tight_layout()
plt.show()

#### Final Training with Best Hyperparameters
We now train a full model for 200 epochs using the hyperparameters selected by the study.

In [ ]:
torch.manual_seed(42)

bp = best.params

heads_final = [OutputHead(dim=2, activation=nn.Tanh, decode=nn.Tanh, name="scaled_output")]
G_final = Generator(
    noise_dim=2,
    num_hidden_layers=2,
    output_heads=heads_final,
    hidden_dims=(128, 128),
    conditional_dim=CONDITIONAL_DIM,
)
D_final = Discriminator(
    feature_dim=2,
    num_hidden_layers=2,
    hidden_dims=(128, 128),
    conditional_dim=CONDITIONAL_DIM,
)

G_losses, D_losses = train_wgan_gp(
    G=G_final, D=D_final,
    X=X_tensor, c=c,
    lr_G=bp["lr_G"],
    lr_D=bp["lr_D"],
    lambda_fm_1=bp["lambda_fm_1"],
    lambda_fm_2=bp["lambda_fm_2"],
    lambda_entropy=bp["lambda_entropy"],
    epochs=200,
    return_history=True,
)

#### Training Loss Curves

In [ ]:
g_epochs, g_vals = zip(*G_losses)
d_epochs, d_vals = zip(*D_losses)

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].plot(g_epochs, g_vals)
axs[0].set_title("Generator Loss")
axs[0].set_xlabel("Epoch")
axs[1].plot(d_epochs, d_vals, color="orange")
axs[1].set_title("Critic Loss")
axs[1].set_xlabel("Epoch")
plt.tight_layout()
plt.show()

#### Visualisation — Real vs Generated

In [ ]:
@torch.no_grad()
def plot_samples(
    G: Generator,
    X_real: np.ndarray,
    c: torch.Tensor,
    c_label: np.ndarray,
    n: int = 2000,
    title: str | None = None,
):
    """
    Plot sample points from the real data and fake data from the generator.

    Parameters
    ----------
    G : Generator
    X_real : np.ndarray  — unscaled real data
    c : torch.Tensor  — conditional (used for generation)
    c_label : np.ndarray  — integer labels (used for coloring)
    n : int  — number of points to plot
    title : str | None
    """
    device = next(G.parameters()).device
    c_in = c[:n].to(device)
    x_fake = G.generate_samples(num_samples=n, c=c_in).cpu().numpy()
    x_fake_unscaled = (x_fake + 1) / 2 * (X_max - X_min) + X_min

    fig, axs = plt.subplots(1, 2, figsize=(10, 5), sharex=True, sharey=True)
    axs[0].scatter(X_real[:n, 0], X_real[:n, 1], c=c_label[:n], alpha=0.3, cmap="tab10")
    axs[0].set_title("Real")
    axs[1].scatter(x_fake_unscaled[:, 0], x_fake_unscaled[:, 1], c=c_label[:n], alpha=0.3, cmap="tab10")
    axs[1].set_title("Generated")
    if title is not None:
        fig.suptitle(title)
    plt.show()

In [ ]:
plot_samples(G_final, X, c, c_label, n=2000, title="Conditional WGAN-GP — Optuna best params")

#### Final Evaluation Score

In [ ]:
final_mse = cluster_mse(G_final, c, c_center, n=4000)
print(f"Final cluster MSE (full training): {final_mse:.6f}")
print(f"Search cluster MSE (best trial):   {best.value:.6f}")